# Introduction to Python

**Topic:** `1.1` &nbsp;|&nbsp; **Module 1: Getting Started with Python**

*Where Python came from, how CPython actually runs your code, and why a language built for readability ended up running most of modern robotics.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M1)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [01_getting_started/README.md](../README.md)

## 1. Why This Topic Matters

Choosing a language is a long-term commitment. Python powers ROS 2 client code, most perception tooling, and every fleet dashboard you will ever be on call for. The interpreter model you learn here also explains every surprising error message you will meet in this course.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Explain CPython's four-stage execution pipeline from source to bytecode.
2. Distinguish name binding from typed assignment, and predict rebinding.
3. Write a PEP 8 script with a docstring and a main guard.
4. Read a traceback and locate the failing line and expression.
5. Explain why robotics orchestration is written in Python.

## 3. Prerequisites

This topic assumes you already have:

- No programming experience assumed.
- Basic familiarity with installing software and using a terminal.

## 4. Mental Model

**Mental model: Python as a very fast, very literal assistant.** You write instructions in order; the assistant reads them top to bottom and does exactly what each line says — including the surprising things. It does not guess intent, so the burden of being precise is on you.

| You are used to | In Python |
| --- | --- |
| A variable is a box with a type | A name is a label for an object |
| Braces define blocks | Indentation defines blocks |
| Code is compiled ahead of time | Code is compiled to bytecode at import |
| Types are declared | Types are dynamic and checked at runtime |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| Interpreter | A program that executes source code instruction by instruction. |
| CPython | The reference Python implementation, written in C. |
| Bytecode | Compact instructions produced by the compiler and run by the VM. |
| REPL | Read-Eval-Print Loop: the interactive prompt. |
| Name binding | Attaching a name to an object with `=`. |
| Dynamic typing | Types are attached to objects and checked at runtime. |
| PEP 8 | The style guide for Python code. |
| Duck typing | If it walks and quacks like a duck, treat it as one. |
| Standard library | The batteries-included modules shipped with Python. |

## 6. Conceptual Explanation

Python is a **general-purpose, high-level programming language** created by Guido van Rossum and first released in 1991. Its design was guided by a single persistent idea: code is read far more often than it is written, so the language should make the *reading* easy. That idea is why Python uses indentation for structure, why its syntax avoids ceremony, and why it ships a large *standard library* rather than pushing everything to third parties.

Python is **interpreted**, or more precisely *compiled to bytecode and then interpreted*. When you run a `.py` file, CPython parses your source into an abstract syntax tree, compiles that into a compact set of instructions, and then executes those instructions in a loop. You never see this bytecode, but it is the reason a Python program starts instantly without a separate build step — and the reason runtime errors and syntax errors currently feel different from each other.

Python is also **dynamically typed**. A name is a *label* bound to an object, not a box with a fixed type. The same name can refer to an integer now and a string a moment later; the type lives on the object, and it is checked while the program runs.

In [ ]:
import sys

print(f"Interpreter : {sys.implementation.name}")
print(f"Version     : {sys.version.split()[0]}")
print(f"Executable  : {sys.executable}")

In [ ]:
value = 42
print(value, type(value))

value = "rover-01"   # rebinding: the name now points at a str
print(value, type(value))

> **Robotics context**
>
> Modern robotics leans on Python for orchestration, perception glue, telemetry and tooling. ROS 2's Python client, numpy, OpenCV bindings and most fleet dashboards are Python. The low-level control loop may be C++, but the code that decides *what the robot should do next* is very often Python.

Three further design decisions shape everything you write. First, **indentation is syntax**: Python removed braces precisely so that the visual structure of a program and its meaning could not drift apart. Second, the language is **batteries-included** — file handling, dates, JSON, sockets and unit testing all ship with the interpreter, so a small robot utility rarely needs a third-party dependency. Third, Python is a **glue language**: it is usually at the top of a stack, calling into C, C++ or Fortran for the parts where raw speed matters, which is exactly the trade-off a robotics stack makes.

One consequence deserves emphasis early. Because there is no compile-time type declaration, a name can silently change type part-way through a program, and nothing will complain until an operation is attempted on the value. That is not sloppiness in the language; it is the price of the flexibility that makes Python productive. Throughout this course you will learn to manage that flexibility deliberately: convert types explicitly at the boundaries where data enters your program, annotate the functions that others depend on, and test the cases where a value might be missing or of an unexpected type.

## 7. Formal Theory

Formally, an **interpreter** executes a program by repeatedly reading a unit of work and performing it, whereas a **compiler** translates a whole program into another language before any of it runs. CPython is a hybrid: it compiles to an intermediate *bytecode* and then interprets that bytecode. The pipeline has four observable stages.

```text
source (.py)  ->  tokens  ->  AST  ->  bytecode  ->  evaluation loop (CPython VM)
```

1. The **lexer** turns characters into tokens (names, numbers, operators).
2. The **parser** builds an Abstract Syntax Tree that encodes structure.
3. The **compiler** emits bytecode — a flat list of instructions.
4. The **evaluation loop** executes those instructions one at a time.

Because the AST is built before execution, *syntax errors* are raised before anything runs, while *runtime errors* (like dividing by zero) happen in the evaluation loop. You can observe the compiled form directly:

In [ ]:
import dis


def cruise_control(target_speed_mps: float) -> float:
    return target_speed_mps * 1.0


dis.dis(cruise_control)

| Term | Meaning |
| --- | --- |
| CPython | The reference implementation written in C. |
| Bytecode | Portable instructions executed by the CPython VM. |
| AST | Tree representation of source structure, built before running. |
| Duck typing | Behaviour matters more than declared type at runtime. |

## 8. Syntax

Python's surface syntax is deliberately small. An **assignment** binds a name to the value on the right with a single `=`. A **statement** occupies a line, and indentation (four spaces is the convention) defines blocks. There are no braces and no mandatory semicolons.

The canonical form:

In [ ]:
# canonical: one statement per line, four-space indentation
robot_name = "rover-01"
max_speed_mps = 1.5
print(robot_name, max_speed_mps)

The anti-pattern — legal, but it fights the language and hides bugs:

```python
robot_name = "rover-01"; max_speed_mps = 1.5;   # semicolons: legal, unidiomatic
print( robot_name ,  max_speed_mps )            # irregular spacing
```

> **Whitespace is significant**
>
> Indentation is part of the grammar, not decoration. Mixing tabs and spaces raises `TabError`; a stray space can silently change which block a line belongs to.

## 9. Basic Examples

**Example 1 — names, values and f-strings.**

In [ ]:
robot = "rover-01"
battery_pct = 87.5
print(f"{robot}: battery at {battery_pct}%")

**Example 2 — arithmetic and division.**

In [ ]:
distance_m = 3.0
speed_mps = 1.5
print(distance_m / speed_mps, "seconds")

## 10. Intermediate Examples

Because typing is dynamic, the *type of a value* is itself something you can inspect at runtime. This one-liner is the fastest way to see Python's dynamic typing in action:

In [ ]:
def describe(obj):
    return f"{obj!r} -> {type(obj).__name__}"


for sample in (7, 3.14, "dock", True, None):
    print(describe(sample))

> **repr vs str**
>
> The `!r` conversion calls `repr()`, which shows a value the way you would write it in code (`'dock'` with quotes). Plain `{}` uses `str()`. In debugging output, prefer `!r`.

## 11. Advanced Examples

Python's bytecode carries a *cache tag* that ties it to a specific CPython version. This is why a `.pyc` compiled by 3.11 is ignored by 3.12 — an engineering detail worth knowing when shipping to a fleet of machines.

In [ ]:
import sys

print("bytecode cache tag:", sys.implementation.cache_tag)
print("supports hash randomisation:", sys.flags.hash_randomization)

## 12. Code Walkthrough

Here is the smallest *useful* ROBO-X bring-up script. It reads the running interpreter, formats a boot banner, and only executes when run as a program — not when imported.

In [ ]:
"""robo_boot.py — the smallest useful ROBO-X bring-up script."""
from __future__ import annotations

import sys

ROBOT_NAME = "rover-01"


def boot_banner(name: str) -> str:
    version = sys.version.split()[0]
    return (
        f"ROBO-X '{name}' booting\n"
        f"  python : {version}\n"
        f"  status : READY"
    )


def main() -> int:
    print(boot_banner(ROBOT_NAME))
    return 0


if __name__ == "__main__":
    main()

## 13. Line-by-Line Explanation

Line by line, matching the walkthrough above:

1. `from __future__ import annotations` makes type hints lazy strings — free forward compatibility for annotations.
2. `import sys` imports the module holding interpreter metadata.
3. `ROBOT_NAME = "rover-01"` is a module-level constant; UPPER_CASE signals "do not rebind".
4. `def boot_banner(name: str) -> str:` declares a function with a parameter and a return annotation.
5. The f-string builds a multi-line banner; `sys.version.split()[0]` extracts just the version number.
6. `main()` is the program entry point; returning an `int` exit code is a Unix convention.
7. The `if __name__ == "__main__":` guard runs `main()` only when the file is executed directly, not when it is imported.

## 14. Common Mistakes

**Mistake 1 — using `=` (assign) where `==` (compare) is meant.**

```text
# wrong: this rebinds battery_pct instead of testing it
if battery_pct = 0:
    print('flat')
```

```python
# right
if battery_pct == 0:
    print('flat')
```

**Mistake 2 — shadowing a built-in name.**

```python
# wrong: 'list' now refers to your variable, not the built-in
list = [1, 2, 3]
more = list((4, 5))        # TypeError: 'list' object is not callable

# right
waypoints = [1, 2, 3]
more = list((4, 5))
```

**Mistake 3 — inconsistent indentation across an editor change.**

```text
# wrong: mixing tabs and spaces in the same block
for i in range(2):
	print(i)        # a tab
        print(i)  # spaces -> TabError
```

```python
# right: four spaces everywhere
for i in range(2):
    print(i)
```

**Mistake 4 — expecting an implicit return.**

```python
# wrong: this returns None
def battery_ok(pct):
    pct > 20

# right
def battery_ok(pct):
    return pct > 20
```

These four share a pattern worth naming: each one is *syntactically* valid, so the interpreter accepts the file without complaint. Nothing warns you at load time that a condition was written as an assignment, that a built-in has been shadowed, that two indentation styles disagree, or that a comparison was evaluated and thrown away. Python will run the program and let the consequences appear later — often far from the line that caused them. This is precisely why the debugging techniques below exist, and why tests that assert on returned values catch defects that reading the source line by line sometimes misses.

## 15. Debugging Techniques

Three techniques you will use constantly, in order of cost:

1. **Type inspection.** Print `type(x)` and `repr(x)` — most beginner bugs are "this is a string when I assumed an int".
2. **Read the traceback bottom-up.** The last line names the exception; the frames above it name the code path.
3. **`breakpoint()`.** Insert `breakpoint()` to drop into pdb and inspect live values.

```python
value = "12"
print(type(value))     # <class 'str'>
print(repr(value))     # '12'
value = int(value)     # now safe to do arithmetic
```

> **Interpreter identity**
>
> `import sys; print(sys.implementation.name)` proves which interpreter runs your code. A surprising number of "Python is broken" reports are really "I am running a different Python than I think".

## 16. Best Practices

- Follow **PEP 8**: four-space indents, `snake_case` for functions and variables, `UPPER_CASE` for constants.
- Give names meaning: `battery_pct` beats `b`.
- Keep one statement per line; drop the semicolons.
- Add short docstrings to modules and functions.
- Prefer small, testable functions over long scripts.
- Never shadow built-ins (`list`, `dict`, `id`, `type`).

## 17. Performance Considerations

At this stage, performance is about *knowing what costs what*, not micro-optimising. Starting the interpreter costs milliseconds; in CPython, attribute lookups and interpreted loop iterations dominate in hot paths. Worry about asymptotics first (an O(n^2) loop over 5,000 telemetry rows matters; a single extra function call does not), and reach for numpy or a C extension only after you have measured.

> **Measure, do not guess**
>
> Use `time.perf_counter()` around the small piece of code you suspect. The `timeit` module automates repeats. Never optimise before measuring.

A concrete illustration. Summing one column of a 5,000-row telemetry file with a Python loop performs roughly one interpreted operation per element; the same arithmetic delegated to a library routine performs the loop inside compiled code. The gap between those two approaches is usually large enough to matter, while the gap between two differently written Python loops on the same data is usually not. Measure which case you are in before spending effort on it, and prefer the clearest implementation until measurement proves otherwise.

## 18. Pythonic Approaches

The same intent, written the way an experienced Python engineer would:

In [ ]:
# Before: manual index bookkeeping
names = ["front", "rear", "left"]
i = 0
while i < len(names):
    print(i, names[i])
    i += 1

# After: iterate directly, with enumerate when you need the index
for index, name in enumerate(names):
    print(index, name)

## 19. Robotics Connection

Python is the language of robotics *glue*: it reads sensors, decides an action, and logs what happened. The course ships a deterministic simulator, so you never need hardware to practise. The snippet below is the smallest interaction with a ROBO-X robot.

In [ ]:
import sys
from pathlib import Path

# Locate the course root (the folder containing `shared/`) and import the sim.
here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "shared").is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name="rover-01", battery_wh=48.0)
print(robot.status())

> **Reproducible by design**
>
> The simulator is seeded, so the readings above are identical on your machine and on mine. That is what makes the exercises gradeable without hardware.

## 20. Engineering Example

A complete, runnable utility that a fleet engineer would actually write: summarise a robot's state into one line for a status dashboard.

In [ ]:
def status_line(state: dict) -> str:
    name = state["name"]
    pct = state["battery_pct"]
    pos = state["position"]
    flag = "LOW" if pct < 20 else "OK"
    return f"{name:<10} {pct:5.1f}%  ({pos[0]:.1f}, {pos[1]:.1f})  {flag}"


sample = {
    "name": "rover-01",
    "battery_pct": 18.4,
    "position": (12.0, 3.5),
}
print(status_line(sample))

## 21. Guided Practice

Follow along in your own notebook. Each step is runnable before the next begins.

1. Bind a variable `robot_name` to a short string and print it.
2. Bind `battery_pct` to a float and print an f-string that includes it.
3. Print `type()` and `repr()` of both variables.
4. Rebind `battery_pct` to the string `"unknown"` and observe that Python allows it — then argue why that is dangerous for a robot.

In [ ]:
# Your turn — complete the steps above.
robot_name = "rover-02"
battery_pct = 91.0
print(f"{robot_name}: {battery_pct}%")
print(type(battery_pct), repr(battery_pct))

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** Why does a Python program start running without a separate build step?
- **Q1.** Renaming a variable to `d` in one function but not another breaks the program. Which principle does this best illustrate?
- **Q1.** Why do the course exercises run against a seeded simulator rather than against real robot hardware?
- **Q1.** In the ROBO-X telemetry controller, why catch `SensorError` and record `None` instead of letting the exception propagate?

## 23. Summary

Python is an interpreted, dynamically typed, general-purpose language built around readability. CPython compiles your source to bytecode and interprets that; names are labels bound to objects; types live on objects and are checked at runtime. These three facts explain almost every beginner surprise — from why syntax errors appear before runtime errors, to why a variable can change type mid-program.

In robotics, Python is the orchestration layer: it reads sensors, makes decisions and logs results, often calling into faster C or C++ code for the tightest control loops. Everything that follows in this course builds on the interpreter model and the naming rules introduced here.

## 24. Key Takeaways

- Python is read-optimised: indentation and small syntax are deliberate.
- CPython compiles to bytecode, then interprets it — errors are either pre-execution (syntax) or in the evaluation loop (runtime).
- Names are labels; objects carry types; typing is dynamic.
- `if __name__ == "__main__":` separates "run as program" from "import".
- PEP 8 names and conventions make code predictable for teams.
- Python is the robotics glue layer; the seeded simulator lets you practise with no hardware.

## 25. Further Exploration

- [The Python Tutorial](https://docs.python.org/3/tutorial/) — official and excellent.
- [PEP 8 — Style Guide](https://peps.python.org/pep-0008/) — the conventions this course follows.
- [Python Developer's Guide: the CPython interpreter](https://devguide.python.org/) — how the interpreter is built.
- [`dis` module docs](https://docs.python.org/3/library/dis.html) — inspect bytecode yourself.
- [The Zen of Python](https://peps.python.org/pep-0020/) — `import this`.